Perfect 👍 Let’s extend your **LangGraph agent framework** with a real **MCP (Model Context Protocol)** setup for weather.

We’ll implement:

* **MCP Weather Server** → serves weather data via `https://wttr.in/{location}?format=3`
* **MCP Weather Client** → the agent/tool uses the MCP server instead of calling requests directly
* **Integration into Agent Factory** → so weather agent gets its tools dynamically

---

# 📂 Updated Project Scaffolding

```
order_mgmt_framework/
│── main.py
│── config/
│   ├── __init__.py
│   ├── settings.py
│── mcp/
│   ├── __init__.py
│   ├── server_weather.py   # Weather MCP Server
│   ├── client_weather.py   # Weather MCP Client
│   ├── server_pollution.py # (optional mock)
│── tools/
│   ├── __init__.py
│   ├── weather_tools.py
│   ├── pollution_tools.py
│── agents/
│   ├── __init__.py
│   ├── agent_factory.py
│   ├── weather_agent.py
│   ├── pollution_agent.py
│   ├── parent_agent.py
```

---

# 🖥️ `mcp/server_weather.py`

MCP server for weather.

```python
from fastapi import FastAPI
import requests

app = FastAPI()

@app.get("/weather/{location}")
def get_weather(location: str):
    """Return weather info for a given location using wttr.in"""
    try:
        response = requests.get(f"https://wttr.in/{location}?format=3", timeout=5)
        return {"location": location, "weather": response.text}
    except Exception as e:
        return {"error": str(e)}
```

👉 Run the MCP server:

```bash
uvicorn mcp.server_weather:app --reload --port 8001
```

---

# 📡 `mcp/client_weather.py`

Client that talks to MCP server.

```python
import requests

BASE_URL = "http://127.0.0.1:8001"

def fetch_weather(location: str) -> str:
    """Calls the MCP weather server to fetch weather."""
    try:
        resp = requests.get(f"{BASE_URL}/weather/{location}", timeout=5)
        data = resp.json()
        if "error" in data:
            return f"Error: {data['error']}"
        return data["weather"]
    except Exception as e:
        return f"Client error: {e}"
```

---

# 🔧 `tools/weather_tools.py`

Uses **MCP client instead of requests** directly.

```python
from mcp.client_weather import fetch_weather

def get_city_weather(city: str) -> str:
    return fetch_weather(city)

def get_country_weather(country: str) -> str:
    return fetch_weather(country)
```

---

# 🏭 `agents/agent_factory.py` (Updated)

```python
from config.settings import AGENT_CONFIG
from tools import weather_tools, pollution_tools

class AgentFactory:
    def __init__(self):
        self.agents = {}

    def build_agent(self, agent_name: str):
        """Dynamically builds an agent from config"""
        if agent_name not in AGENT_CONFIG:
            raise ValueError(f"Unknown agent: {agent_name}")

        config = AGENT_CONFIG[agent_name]
        tool_funcs = []

        # Pick tools based on agent config
        for tool in config.get("tools", []):
            if hasattr(weather_tools, tool):
                tool_funcs.append(getattr(weather_tools, tool))
            elif hasattr(pollution_tools, tool):
                tool_funcs.append(getattr(pollution_tools, tool))

        def run(task: dict):
            city = task.get("city")
            country = task.get("country")
            for tool in tool_funcs:
                if city and "city" in tool.__name__:
                    return tool(city)
                if country and "country" in tool.__name__:
                    return tool(country)
            return f"{agent_name} agent: No location provided"

        self.agents[agent_name] = run
        return run

    def get_agent(self, agent_name: str):
        return self.agents.get(agent_name) or self.build_agent(agent_name)
```

---

# 🚀 `main.py` (Run Parent Agent + Weather MCP)

```python
from langgraph.graph import StateGraph, START, END
from agents.agent_factory import AgentFactory
from agents.parent_agent import parent_router

def build_graph():
    factory = AgentFactory()
    weather_agent = factory.get_agent("weather")
    pollution_agent = factory.get_agent("pollution")

    builder = StateGraph(dict)

    builder.add_node("parent_router", lambda state: {"next": parent_router(state["prompt"])})
    builder.add_node("weather", lambda state: {"result": weather_agent(state)})
    builder.add_node("pollution", lambda state: {"result": pollution_agent(state)})

    builder.add_edge(START, "parent_router")
    builder.add_conditional_edges(
        "parent_router",
        lambda out: out["next"],
        {"weather": "weather", "pollution": "pollution", "end": END}
    )
    builder.add_edge("weather", END)
    builder.add_edge("pollution", END)

    return builder.compile()

def main():
    graph = build_graph()

    prompts = [
        {"prompt": "What's the weather in Paris?", "city": "Paris"},
        {"prompt": "Check pollution in Delhi", "city": "Delhi"},
    ]

    for p in prompts:
        print(f"\n📝 Prompt: {p['prompt']}")
        result = graph.invoke(p)
        print("🤖 Response:", result.get("result"))

if __name__ == "__main__":
    main()
```

---

✅ **Flow**:

1. Run MCP Weather Server (`uvicorn mcp.server_weather:app --reload --port 8001`)
2. Start framework (`python main.py`)
3. Parent routes query → weather agent → MCP client → MCP server → `wttr.in`

---

Do you want me to also make **Pollution Agent** run with a **mock MCP server** (like `/pollution/{city}`) so both weather & pollution agents work end-to-end via MCP?
